In [1]:
# data preparation


import numpy as np
import os
import time
from numba import njit
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ==========================================
# 1. parameters setting
# ==========================================
J1_map = {('A', 'A'): 1.0,  ('B', 'B'): 0.5,  ('A', 'B'): 0.8}
J2_map = {('A', 'A'): -0.45,('B', 'B'): -0.25,('A', 'B'): -0.35}

def get_coupling_obc(seq_str):
    N = len(seq_str)
    J1, J2 = np.zeros(N), np.zeros(N)
    for i in range(N - 1): J1[i] = J1_map[tuple(sorted([seq_str[i], seq_str[i+1]]))]
    for i in range(N - 2): J2[i] = J2_map[tuple(sorted([seq_str[i], seq_str[i+2]]))]
    return J1, J2

# ====================================================================
# 2. Numba  (OBC + NoGIL)
# ====================================================================
@njit(nogil=True)
def calc_Heff_obc(i, spins, J1, J2):
    N = spins.shape[0]
    Hx, Hy, Hz = 0.0, 0.0, 0.0
    if i > 0:
        Hx += J1[i-1]*spins[i-1,0]; Hy += J1[i-1]*spins[i-1,1]; Hz += J1[i-1]*spins[i-1,2]
    if i < N - 1:
        Hx += J1[i]*spins[i+1,0];   Hy += J1[i]*spins[i+1,1];   Hz += J1[i]*spins[i+1,2]
    if i > 1:
        Hx += J2[i-2]*spins[i-2,0]; Hy += J2[i-2]*spins[i-2,1]; Hz += J2[i-2]*spins[i-2,2]
    if i < N - 2:
        Hx += J2[i]*spins[i+2,0];   Hy += J2[i]*spins[i+2,1];   Hz += J2[i]*spins[i+2,2]
    return np.array([Hx, Hy, Hz])

@njit(nogil=True)
def calc_total_energy_obc(spins, J1, J2):
    N = spins.shape[0]; E = 0.0
    for i in range(N - 1): E -= J1[i] * (spins[i,0]*spins[i+1,0] + spins[i,1]*spins[i+1,1] + spins[i,2]*spins[i+1,2])
    for i in range(N - 2): E -= J2[i] * (spins[i,0]*spins[i+2,0] + spins[i,1]*spins[i+2,1] + spins[i,2]*spins[i+2,2])
    return E

@njit(nogil=True)
def sample_von_Mises_Fisher(mu_dir, kappa):
    if kappa < 1e-8:
        vec = np.random.randn(3); return vec / np.linalg.norm(vec)
    u = np.random.rand()
    z = 1.0 + (1.0 / kappa) * np.log(u + (1.0 - u) * np.exp(-2.0 * kappa))
    z = min(1.0, max(-1.0, z))
    phi = np.random.rand() * 2 * np.pi
    r = np.sqrt(1.0 - z**2)
    local_vec = np.array([r * np.cos(phi), r * np.sin(phi), z])
    mu_norm = np.linalg.norm(mu_dir); mu_dir = mu_dir / mu_norm
    if mu_dir[2] > 0.9999: return local_vec
    if mu_dir[2] < -0.9999: return np.array([local_vec[0], -local_vec[1], -local_vec[2]])
    v1 = np.array([-mu_dir[1], mu_dir[0], 0.0]); v1 = v1 / np.linalg.norm(v1)
    v2 = np.array([mu_dir[1]*v1[2] - mu_dir[2]*v1[1], mu_dir[2]*v1[0] - mu_dir[0]*v1[2], mu_dir[0]*v1[1] - mu_dir[1]*v1[0]])
    global_vec = local_vec[0]*v1 + local_vec[1]*v2 + local_vec[2]*mu_dir
    return global_vec / np.linalg.norm(global_vec)

@njit(nogil=True)
def sweep_obc(spins, J1, J2, beta):
    N = spins.shape[0]
    for i in range(N):
        H_eff = calc_Heff_obc(i, spins, J1, J2)
        h_mag = np.linalg.norm(H_eff)
        if h_mag > 1e-8:
            if np.random.rand() < 0.5: spins[i] = sample_von_Mises_Fisher(H_eff, beta * h_mag)
            else:
                mu_dir = H_eff / h_mag
                dot_val = spins[i,0]*mu_dir[0] + spins[i,1]*mu_dir[1] + spins[i,2]*mu_dir[2]
                spins[i] = 2.0 * dot_val * mu_dir - spins[i]
                spins[i] /= np.linalg.norm(spins[i])
    return spins

@njit(nogil=True)
def generate_snapshots_for_seq(J1, J2, num_snapshots, burn_in=2000, thinning=100):
    N = J1.shape[0]; M = 16
    temperatures = np.zeros(M)
    for i in range(M): temperatures[i] = 10**(np.log10(5.0) + i * (np.log10(0.05) - np.log10(5.0)) / (M - 1))
    betas = 1.0 / temperatures
    
    replicas = np.zeros((M, N, 3))
    for m in range(M):
        for i in range(N):
            vec = np.random.randn(3); replicas[m, i] = vec / np.linalg.norm(vec)
            
    energies = np.zeros(M)
    for m in range(M): energies[m] = calc_total_energy_obc(replicas[m], J1, J2)
        
    for step in range(burn_in):
        for m in range(M):
            replicas[m] = sweep_obc(replicas[m], J1, J2, betas[m])
            energies[m] = calc_total_energy_obc(replicas[m], J1, J2)
        if step % 10 == 0:
            for m in range(M - 1):
                delta_beta = betas[m+1] - betas[m]
                delta_E = energies[m+1] - energies[m]
                if delta_E * delta_beta > 0 or np.random.rand() < np.exp(delta_beta * delta_E):
                    temp_rep = replicas[m].copy(); replicas[m] = replicas[m+1]; replicas[m+1] = temp_rep
                    temp_E = energies[m]; energies[m] = energies[m+1]; energies[m+1] = temp_E
                    
    snapshots = np.zeros((num_snapshots, N, 3))
    for snap_idx in range(num_snapshots):
        for step in range(thinning):
            for m in range(M):
                replicas[m] = sweep_obc(replicas[m], J1, J2, betas[m])
                energies[m] = calc_total_energy_obc(replicas[m], J1, J2)
            if step % 10 == 0:
                for m in range(M - 1):
                    delta_beta = betas[m+1] - betas[m]
                    delta_E = energies[m+1] - energies[m]
                    if delta_E * delta_beta > 0 or np.random.rand() < np.exp(delta_beta * delta_E):
                        temp_rep = replicas[m].copy(); replicas[m] = replicas[m+1]; replicas[m+1] = temp_rep
                        temp_E = energies[m]; energies[m] = energies[m+1]; energies[m+1] = temp_E
        snapshots[snap_idx] = replicas[-1].copy()
    return snapshots

@njit(nogil=True)
def compute_theoretical_tangent_forces(spins, J1, J2):
    N = spins.shape[0]
    F_tan = np.zeros_like(spins)
    for i in range(N):
        H_eff = calc_Heff_obc(i, spins, J1, J2)
        radial_component = H_eff[0]*spins[i,0] + H_eff[1]*spins[i,1] + H_eff[2]*spins[i,2]
        F_tan[i, 0] = H_eff[0] - radial_component * spins[i, 0]
        F_tan[i, 1] = H_eff[1] - radial_component * spins[i, 1]
        F_tan[i, 2] = H_eff[2] - radial_component * spins[i, 2]
    return F_tan

# ==========================================
# 3. Worker function
# ==========================================
def worker_task(args):
    seq_str, snapshots_per_seq, include_forces = args
    J1, J2 = get_coupling_obc(seq_str)
    
    snapshots = generate_snapshots_for_seq(J1, J2, snapshots_per_seq, burn_in=2000, thinning=100)
        
    forces = None
    if include_forces:
        forces = np.zeros_like(snapshots, dtype=np.float32)
        for snap in range(snapshots_per_seq):
            forces[snap] = compute_theoretical_tangent_forces(snapshots[snap], J1, J2)
            
    return seq_str, snapshots.astype(np.float32), forces

# ==========================================
# 4. save
# ==========================================
def generate_dataset_chunked(num_seqs, snapshots_per_seq, dataset_dir, chunk_size=1000, include_forces=False):
    print(f"\n🚀 category: {dataset_dir} | total: {num_seqs} seq | each chunk: {chunk_size} seq")
    start_time = time.time()
    
    os.makedirs(dataset_dir, exist_ok=True)
    
    print("🔥 preparing Numba...")
    _J1, _J2 = get_coupling_obc("ABAB" * 5)
    _ = generate_snapshots_for_seq(_J1, _J2, 2, burn_in=10, thinning=10)
    
    print("🧬 generating unique sequences ( 100% no duplication)...")
    np.random.seed(42)
    unique_seqs = set()
    while len(unique_seqs) < num_seqs:
        N = np.random.randint(20, 51)
        seq_array = np.random.choice(['A', 'B'], size=N)
        unique_seqs.add("".join(seq_array))
    
    seq_list = list(unique_seqs)
    num_cores = os.cpu_count()
    
    num_chunks = int(np.ceil(num_seqs / chunk_size))
    print(f"📦 totally {num_chunks} chunks，preparing calculation (by {num_cores} cores)...")
    
    for chunk_idx in range(num_chunks):
        chunk_filename = os.path.join(dataset_dir, f"chunk_{chunk_idx:04d}.npz")
        
        # skip if chunk exist！
        if os.path.exists(chunk_filename):
            print(f"⏩ skip {chunk_filename} (it existed!)")
            continue
            
        print(f"\n⏳ calculating Chunk {chunk_idx:04d}/{num_chunks-1} ...")
        
        start_idx = chunk_idx * chunk_size
        end_idx = min(start_idx + chunk_size, num_seqs)
        current_chunk_seqs = seq_list[start_idx:end_idx]
        
        tasks = [(seq, snapshots_per_seq, include_forces) for seq in current_chunk_seqs]
        
        dataset_seqs = []
        dataset_spins = []
        dataset_forces = []
        
        with ThreadPoolExecutor(max_workers=num_cores) as executor:
            futures = {executor.submit(worker_task, task): task for task in tasks}
            

            with tqdm(total=len(tasks), desc=f"Chunk {chunk_idx}", unit="seq", 
                      mininterval=2.0, maxinterval=5.0) as pbar:
                for future in as_completed(futures):
                    seq_str, spins, forces = future.result()
                    dataset_seqs.append(seq_str)
                    dataset_spins.append(spins)
                    if include_forces: dataset_forces.append(forces)
                    pbar.update(1)
                    
        seqs_arr = np.array(dataset_seqs)
        spins_arr = np.empty(len(dataset_spins), dtype=object)
        for i, arr in enumerate(dataset_spins):
            spins_arr[i] = arr
            
        save_dict = {'sequences': seqs_arr, 'spins': spins_arr}
        
        if include_forces:
            forces_arr = np.empty(len(dataset_forces), dtype=object)
            for i, arr in enumerate(dataset_forces):
                forces_arr[i] = arr
            save_dict['tangent_forces'] = forces_arr
            
        np.savez_compressed(chunk_filename, **save_dict)
        print(f"💾 {chunk_filename} is written！")
        
    total_time = (time.time() - start_time) / 3600
    print(f"\n🎉 all date generated！category: {dataset_dir} | time: {total_time:.2f} hours")

if __name__ == '__main__':
    # 1. 生成训练集 (40,000 seq, divided into 40 chunk in SO3_TrainSet_40k)
    generate_dataset_chunked(
        num_seqs=40000, 
        snapshots_per_seq=10, 
        dataset_dir="SO3_TrainSet_40k", 
        chunk_size=1000,        # save per 1000 seq！
        include_forces=False
    )
    
    # 2. 生成测试集 (2,000 seq, divided into 2 chunk in SO3_TestSet_2k_with_OracleForces)
    generate_dataset_chunked(
        num_seqs=2000, 
        snapshots_per_seq=5, 
        dataset_dir="SO3_TestSet_2k_with_OracleForces", 
        chunk_size=1000, 
        include_forces=True
    )


🚀 category: SO3_TrainSet_40k | total: 40000 seq | each chunk: 1000 seq
🔥 preparing Numba...
🧬 generating unique sequences ( 100% no duplication)...
📦 totally 40 chunks，preparing calculation (by 32 cores)...
⏩ skip SO3_TrainSet_40k\chunk_0000.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0001.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0002.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0003.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0004.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0005.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0006.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0007.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0008.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0009.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0010.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0011.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0012.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0013.npz (it existed!)
⏩ skip SO3_TrainSet_40k\chunk_0014.npz (it existed

In [ ]:
# train model
import train_real_data as trd
trd.train()

In [1]:
# model analysis
import torch
import numpy as np
import glob
from mini_af3_model import MiniAF3ScoreModel

# real parameters
J_TRUE = {
    'J1_AA': 1.0, 'J1_BB': 0.5, 'J1_AB': 0.8,
    'J2_AA': -0.45,'J2_BB': -0.25,'J2_AB': -0.35
}

def build_geometric_design_matrix(seq_str, spins):

    N = len(seq_str)
    X = np.zeros((N, 3, 6)) # basis for 6 parameters 
    
    # index for 6 parameters
    idx_map = {'J1_AA': 0, 'J1_BB': 1, 'J1_AB': 2, 
               'J2_AA': 3, 'J2_BB': 4, 'J2_AB': 5}
    
    for i in range(N):
        #  (H_eff)
        H_basis = np.zeros((6, 3))
        
        # J1 (r=1)
        if i > 0:
            pair = "".join(sorted([seq_str[i], seq_str[i-1]]))
            H_basis[idx_map[f'J1_{pair}']] += spins[i-1]
        if i < N - 1:
            pair = "".join(sorted([seq_str[i], seq_str[i+1]]))
            H_basis[idx_map[f'J1_{pair}']] += spins[i+1]
            
        # J2 (r=2)
        if i > 1:
            pair = "".join(sorted([seq_str[i], seq_str[i-2]]))
            H_basis[idx_map[f'J2_{pair}']] += spins[i-2]
        if i < N - 2:
            pair = "".join(sorted([seq_str[i], seq_str[i+2]]))
            H_basis[idx_map[f'J2_{pair}']] += spins[i+2]
            
        # (Projection)
        # force should on O3
        for m in range(6):
            radial = np.dot(H_basis[m], spins[i])
            X[i, :, m] = H_basis[m] - radial * spins[i]
            
    return X

def run_phase_2_inversion(model_path="mini_af3_epoch_6.pt", test_dir="SO3_TestSet_2k_with_OracleForces"):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 start OLS | device: {device}")
    
    # 1. load ai oracle
    model = MiniAF3ScoreModel(c_s=64, c_z=32, num_blocks=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval() # freeze model
    
    print(f"📂 loading test data (Thermal Probes)...")
    files = sorted(glob.glob(f"{test_dir}/chunk_*.npz"))
    
    Y_all = [] #  AI prediction
    X_all = [] # geometric matrix
    
    with torch.no_grad():
        for f in files:
            data = np.load(f, allow_pickle=True)
            seqs, spins_list = data['sequences'], data['spins']
            
            for seq_str, snapshots in zip(seqs, spins_list):
                N = len(seq_str)
                seq_tensor = torch.tensor([0 if c == 'A' else 1 for c in seq_str], dtype=torch.long).unsqueeze(0).to(device)
                mask = torch.ones((1, N), dtype=torch.bool).to(device)
                
                # set t->0 
                t_zero = torch.ones((1, 1)).to(device) * 1e-4 
                

                for snap in snapshots:
                    spin_tensor = torch.tensor(snap, dtype=torch.float32).unsqueeze(0).to(device)
                    
                    # 【query AI oracle】
                    ai_score = model(seq_tensor, spin_tensor, t_zero, mask).squeeze(0).cpu().numpy()
                    
                    # AI predict Y 
                    Y_all.append(ai_score.flatten()) 
                    
                    # 【geometric matrix】
                    X_basis = build_geometric_design_matrix(seq_str, snap)
                    # X reshape (3N, 6)
                    X_all.append(X_basis.reshape(3 * N, 6))
                    
            print(f"  scan: {f}")

# ==========================================
    #  (OLS)
    # ==========================================
    print("\n⚔️ start (Ordinary Least Squares)...")
    Y_massive = np.concatenate(Y_all, axis=0) # (3 * N * Total_Snapshots,)
    X_massive = np.concatenate(X_all, axis=0) # (3 * N * Total_Snapshots, 6)
    
    print(f"📊 matrix sacle: Y dim {Y_massive.shape}, X dim {X_massive.shape}")
    
    # solve J_model_raw = (X^T X)^(-1) X^T Y
    J_model_raw, residuals, rank, s = np.linalg.lstsq(X_massive, Y_massive, rcond=None)
    
    # ----------------------------------------------------
    # (Scale-Invariant Metrics)
    # 
    # ----------------------------------------------------
    
    # 1. compare predicted J and real J
    labels = ['J1_AA', 'J1_BB', 'J1_AB', 'J2_AA', 'J2_BB', 'J2_AB']
    J_true_vec = np.array([J_TRUE[l] for l in labels])
    J_pred_vec = J_model_raw  
    
    # 2. R^2 calculation
    ss_res = np.sum((Y_massive - X_massive @ J_model_raw)**2)
    ss_tot = np.sum((Y_massive - np.mean(Y_massive))**2)
    r_squared = 1 - (ss_res / ss_tot)
    
    # 3. Cosine Similarity = (A dot B) / (||A|| * ||B||)
    dot_product = np.dot(J_true_vec, J_pred_vec)
    norm_true = np.linalg.norm(J_true_vec)
    norm_pred = np.linalg.norm(J_pred_vec)
    cos_sim = dot_product / (norm_true * norm_pred)
    
    
    # set J1_AA = 1.0
    norm_factor_true = J_true_vec[0]  
    norm_factor_pred = J_pred_vec[0]  
    
    J_true_normalized = J_true_vec / norm_factor_true
    J_pred_normalized = J_pred_vec / norm_factor_pred

    # ==========================================
    # print
    # ==========================================
    print("\n" + "="*60)
    print("🏆 PHASE II: SCALE-INVARIANT PHYSICAL JUDGEMENT 🏆")
    print("="*60)
    
    print(f"\n[result 1] R^2 (Linear Subspace Isomorphism):")
    print(f"         R^2 = {r_squared:.4f}")

    
    print(f"\n[result 2] (Geometric Topology Equivalence):")
    print(f"         S_cos = {cos_sim:.4f} ({cos_sim * 100:.2f}%)")

    
    print("\n[result 3] relative ratio (set J1_AA = 1.00 ):")
    print(f"{'para':<10} | {'True Ratio':<15} | {'AI Pred Ratio':<15} | {'abs err':<10}")
    print("-" * 60)
    
    for i, label in enumerate(labels):
        t_val = J_true_normalized[i]
        p_val = J_pred_normalized[i]
        err = abs(t_val - p_val) 
        print(f"{label:<10} | {t_val:>12.4f}    | {p_val:>12.4f}    | {err:>8.4f}")
        

if __name__ == '__main__':
    # 请替换为你的模型路径
    run_phase_2_inversion(model_path="mini_af3_epoch_20.pt")

🚀 start OLS | device: cuda
📂 loading test data (Thermal Probes)...
  scan: SO3_TestSet_2k_with_OracleForces\chunk_0000.npz
  scan: SO3_TestSet_2k_with_OracleForces\chunk_0001.npz

⚔️ start (Ordinary Least Squares)...
📊 matrix sacle: Y dim (1057440,), X dim (1057440, 6)

🏆 PHASE II: SCALE-INVARIANT PHYSICAL JUDGEMENT 🏆

[result 1] R^2 (Linear Subspace Isomorphism):
         R^2 = 0.8698

[result 2] (Geometric Topology Equivalence):
         S_cos = 0.9974 (99.74%)

[result 3] relative ratio (set J1_AA = 1.00 ):
para       | True Ratio      | AI Pred Ratio   | abs err   
------------------------------------------------------------
J1_AA      |       1.0000    |       1.0000    |   0.0000
J1_BB      |       0.5000    |       0.6206    |   0.1206
J1_AB      |       0.8000    |       0.8759    |   0.0759
J2_AA      |      -0.4500    |      -0.4643    |   0.0143
J2_BB      |      -0.2500    |      -0.2928    |   0.0428
J2_AB      |      -0.3500    |      -0.3778    |   0.0278
